# Презентация: Data Analytics — ICH IT School

In [52]:
import os
from pptx import Presentation
from pptx.util import Inches, Pt, Emu
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN
import pptx.oxml.ns as nsmap
from lxml import etree
import pandas as pd
import numpy as np
from pptx.util import Inches as PptxInches

import help_130625_dam as h


In [53]:
#  Цветовая схема 
C_DARK   = RGBColor(0x0F, 0x34, 0x60)   # глубокий синий (фон заголовков)
C_TEAL   = RGBColor(0x21, 0xA6, 0xF5)   # небесно-синий (акцент)
C_WHITE  = RGBColor(0xFF, 0xFF, 0xFF)   # белый (текст на темном фоне)
C_LIGHT  = RGBColor(0xE8, 0xF4, 0xFF)   # очень светлый голубой фон
C_GRAY   = RGBColor(0x4A, 0x5E, 0x7A)   # сине-серый вспомогательный текст
C_GREEN  = RGBColor(0x1A, 0xB3, 0x7B)   # успех / ✓
C_ORANGE = RGBColor(0xF5, 0x8A, 0x1E)   # предупреждение

SLIDE_W = Inches(13.33)
SLIDE_H = Inches(7.5)

prs = Presentation()
prs.slide_width  = SLIDE_W
prs.slide_height = SLIDE_H

BLANK = prs.slide_layouts[6]   # пустой макет

def add_rect(slide, l, t, w, h, color, radius=None):
    """Добавляет прямоугольник заданного цвета."""
    shape = slide.shapes.add_shape(1, Inches(l), Inches(t), Inches(w), Inches(h))
    shape.fill.solid()
    shape.fill.fore_color.rgb = color
    shape.line.fill.background()
    return shape

def add_text(slide, text, l, t, w, h,
             size=18, bold=False, color=C_DARK,
             align=PP_ALIGN.LEFT, wrap=True):
    """Добавляет текстовый блок."""
    txb = slide.shapes.add_textbox(Inches(l), Inches(t), Inches(w), Inches(h))
    tf  = txb.text_frame
    tf.word_wrap = wrap
    p   = tf.paragraphs[0]
    p.alignment = align
    run = p.add_run()
    run.text = text
    run.font.size  = Pt(size)
    run.font.bold  = bold
    run.font.color.rgb = color
    return txb

def add_bullet_box(slide, items, l, t, w, h,
                   size=16, color=C_DARK, spacing=0.55):
    """Рисует список с иконкой ▸ перед каждым пунктом."""
    txb = slide.shapes.add_textbox(Inches(l), Inches(t), Inches(w), Inches(h))
    tf  = txb.text_frame
    tf.word_wrap = True
    for i, item in enumerate(items):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.space_before = Pt(spacing * 6)
        run = p.add_run()
        run.text = f'▸  {item}'
        run.font.size  = Pt(size)
        run.font.color.rgb = color
    return txb

def add_header_bar(slide, title, subtitle=None):
    """Стандартная синяя шапка с заголовком."""
    add_rect(slide, 0, 0, 13.33, 1.4, C_DARK)
    add_rect(slide, 0, 1.4, 0.06, 6.1, C_TEAL)   # вертикальная полоска-акцент
    add_text(slide, title, 0.35, 0.18, 12.0, 0.7,
             size=30, bold=True, color=C_WHITE)
    if subtitle:
        add_text(slide, subtitle, 0.35, 0.82, 12.0, 0.45,
                 size=16, color=C_TEAL)

def bg_light(slide):
    """Светлый голубой фон слайда."""
    add_rect(slide, 0, 0, 13.33, 7.5, C_LIGHT)

## Слайд 1 — Титульный

In [54]:
slide = prs.slides.add_slide(BLANK)

# Фон: тёмный низ + светлый верх
add_rect(slide, 0,   0, 13.33, 4.5,  C_DARK)
add_rect(slide, 0, 4.5, 13.33, 3.0,  C_LIGHT)

# Декоративная бирюзовая полоса
add_rect(slide, 0, 4.3, 13.33, 0.2, C_TEAL)

logo = add_rect(slide, 0.6, 0.5, 1.6, 0.65, C_TEAL)
add_text(slide, 'ICH IT SCHOOL', 0.65, 0.52, 1.5, 0.55,
         size=13, bold=True, color=C_DARK)

# Главный заголовок
add_text(slide, 'Data Analytics', 0.6, 1.4, 12.0, 1.0,
         size=52, bold=True, color=C_WHITE)
add_text(slide, 'Очистка и анализ данных из CRM системы для повышения эффективности работы \n'
    'онлайн-школы программирования.',
         0.6, 2.5, 11.5, 0.7, size=22, color=C_TEAL)

# Метаданные
add_text(slide, 'Данные: CRM-выгрузки за период с июля 2023 до июля 2024 (Deals, Contacts, Calls, Spend)',
         0.6, 5.0, 11.0, 0.5, size=14, color=C_GRAY)
add_text(slide, 'Data Analytics  •  Veremeienko Oleksandr  •  130625-dam',
         0.6, 5.6, 10.0, 0.5, size=13, color=C_GRAY)
print('Слайд 1 добавлен: Титульный')


Слайд 1 добавлен: Титульный


## Слайд 2 — Обзор датасетов

In [55]:
slide = prs.slides.add_slide(BLANK)
bg_light(slide)
add_header_bar(slide, 'Источники данных',
               'Четыре CRM-выгрузки — основа всего анализа')

# 4 карточки: актуальные числа из сырых файлов
cards = [
    ('Contacts', '18 548 строк', [
        'Лиды и клиенты ICH IT School',
        'Менеджеры по продажам',
        'Дата регистрации в CRM',
        'Дата изменения контакта',
    ]),
    ('Deals', '21 595 строк', [
        'Сделки и офёрты (с июля 2023)',
        'Продукт, сумма, тип оплаты',
        'Стадия CRM-воронки (13 стадий)',
        'SLA, уровень языка, город',
    ]),
    ('Spend', '20 779 строк', [
        'Рекламные расходы 2023-07-03 - 2024-06-21',
        'Кампания / группа / объявление',
        'Клики и показы',
        '14 источников трафика',
    ]),
    ('Calls', '95 874 строки', [
        'Звонки менеджеров',
        'Входящие / исходящие / пропущенные',
        'Длительность и итоговый статус',
        'Привязка к контакту',
    ]),
]

card_w, card_h = 2.9, 4.8
for i, (name, rows, bullets) in enumerate(cards):
    x = 0.35 + i * 3.24
    y = 1.55
    add_rect(slide, x, y, card_w, card_h, C_WHITE)
    add_rect(slide, x, y, card_w, 0.65, C_DARK)
    add_rect(slide, x, y + 0.65, 0.06, card_h - 0.65, C_TEAL)
    add_text(slide, name, x + 0.15, y + 0.1, card_w - 0.2, 0.5,
             size=18, bold=True, color=C_WHITE)
    add_text(slide, rows, x + 0.15, y + 0.72, card_w - 0.2, 0.4,
             size=13, bold=False, color=C_TEAL)
    add_bullet_box(slide, bullets,
                   x + 0.18, y + 1.15, card_w - 0.25, card_h - 1.3,
                   size=13, color=C_DARK)

print('Слайд 2 добавлен: Источники данных')


Слайд 2 добавлен: Источники данных


## Слайд 3 — Процесс очистки (обзор)

In [ ]:
slide = prs.slides.add_slide(BLANK)
bg_light(slide)
add_header_bar(slide, 'Процесс очистки данных',
               'Единый 5-шаговый pipeline для всех датасетов')

# 5 шагов pipeline с кратким описанием «зачем»
steps = [
    ('1', 'Загрузка\n& осмотр',      'shape, dtypes,\nnulls, uniques'),
    ('2', 'Дедуп-\nликация',         'полные + бизнес-\nдубликаты'),
    ('3', 'Типы\nданных',            'даты → dt64\nID → Int64'),
    ('4', 'Нормали-\nзация',         'стадии → 4 гр.\nязык → 7 ур.'),
    ('5', 'Сохранение\n+ связи',     '.pkl + contact_id\nкак единый ключ'),
]

for i, (num, title, detail) in enumerate(steps):
    x = 0.5 + i * 2.55
    y = 2.0
    circle = slide.shapes.add_shape(
        9, Inches(x + 0.9), Inches(y), Inches(0.7), Inches(0.7))
    circle.fill.solid()
    circle.fill.fore_color.rgb = C_TEAL
    circle.line.fill.background()
    add_text(slide, num, x + 0.9, y + 0.05, 0.7, 0.6,
             size=20, bold=True, color=C_WHITE, align=PP_ALIGN.CENTER)
    add_text(slide, title, x + 0.2, y + 0.85, 2.0, 0.8,
             size=15, bold=True, color=C_DARK, align=PP_ALIGN.CENTER)
    add_text(slide, detail, x + 0.2, y + 1.65, 2.0, 0.7,
             size=12, color=C_GRAY, align=PP_ALIGN.CENTER)
    if i < len(steps) - 1:
        add_text(slide, '→', x + 2.25, y + 0.1, 0.4, 0.5,
                 size=28, bold=True, color=C_TEAL, align=PP_ALIGN.CENTER)

# Итоговая строка с реальными числами
add_rect(slide, 0.5, 5.0, 12.33, 0.05, C_TEAL)
add_text(slide,
         'Результат: 5 чистых pkl-файла   |   '
         'Contacts 18 510  ·  Deals 19 815  ·  Spend 19 862  ·  Calls 92 599  ·  Merging',
         0.5, 5.15, 12.5, 0.5, size=14, color=C_DARK, align=PP_ALIGN.CENTER)

# Блок «зачем это нужно»
add_rect(slide, 0.5, 5.8, 12.33, 1.35, C_WHITE)
add_rect(slide, 0.5, 5.8, 0.06, 1.35, C_TEAL)
add_text(slide,
         'Без очистки: Искажение данных при анализе, ошибки при объединении '
         'агрегации  ·  215 вариантов уровня языка → нельзя группировать  ·  '
         'дубли → завышенное количество и заниженная статистика',
         0.7, 5.9, 12.0, 1.15, size=13, color=C_DARK)

print('Слайд 3 добавлен: Pipeline очистки')


Слайд 3 добавлен: Pipeline очистки


## Слайд 4 — Contacts

In [57]:
slide = prs.slides.add_slide(BLANK)
bg_light(slide)
add_header_bar(slide, 'Очистка: Contacts',
               'Справочник лидов и клиентов — 18 548 строк исходно')

C_RED = RGBColor(0xC0, 0x39, 0x2B)
C_AMB = RGBColor(0xD3, 0x5A, 0x00)

BW, BH = 6.2, 1.92
CX1, CX2 = 0.35, 6.78
RY1, RY2 = 1.65, 3.73

# Box 1: Что случилось 
add_rect(slide, CX1, RY1, BW, 0.36, C_DARK)
add_text(slide, 'Что случилось', CX1 + 0.1, RY1 + 0.03, BW - 0.15, 0.3,
         size=13, bold=True, color=C_WHITE)
add_bullet_box(slide, [
    '38 полных дублирующихся строк в таблице',
    'Менеджер с именем "False" — CRM-мусор',
    'Поля дат хранились как object (строки)',
    'ID контакта имел тип float64',
], CX1 + 0.12, RY1 + 0.42, BW - 0.2, BH - 0.5, size=12, color=C_DARK)

#  Box 2: Почему случилось 
add_rect(slide, CX2, RY1, BW, 0.36, C_AMB)
add_text(slide, 'Почему случилось', CX2 + 0.1, RY1 + 0.03, BW - 0.15, 0.3,
         size=13, bold=True, color=C_WHITE)
add_bullet_box(slide, [
    'CRM не блокирует повторные заявки одного лида',
    'Технический пользователь "False" в системе-источнике',
    'CRM экспортирует даты в формате строки',
    'ID выгружается как дробное число. Проблема объединения',
], CX2 + 0.12, RY1 + 0.42, BW - 0.2, BH - 0.5, size=12, color=C_DARK)

#  Box 3: Если ничего не делать 
add_rect(slide, CX1, RY2, BW, 0.36, C_RED)
add_text(slide, 'Если ничего не делать', CX1 + 0.1, RY2 + 0.03, BW - 0.15, 0.3,
         size=13, bold=True, color=C_WHITE)
add_bullet_box(slide, [
    'Двойной учёт лидов → конверсия завышена или занижена',
    'Менеджер "False" попадает в отчёты по продажам',
    'Операции c датами падают с ошибкой',
    'При объединении со сделками и звонками контакты не совпадают → теряются данные',
], CX1 + 0.12, RY2 + 0.42, BW - 0.2, BH - 0.5, size=12, color=C_DARK)

#  Box 4: Что сделали 
add_rect(slide, CX2, RY2, BW, 0.36, C_TEAL)
add_text(slide, 'Что сделали', CX2 + 0.1, RY2 + 0.03, BW - 0.15, 0.3,
         size=13, bold=True, color=C_DARK)
add_bullet_box(slide, [
    'Удалено 38 дублей',
    'Менеджер "False" удален с подменой на Jane Smith',
    'Даты преобразованы в datetime64[us]',
    'ID → Int64; 38 контактов перепривязаны в Deals/Calls',
    'Результат: 18 510 уникальных чистых записей',
], CX2 + 0.12, RY2 + 0.42, BW - 0.2, BH - 0.5, size=12, color=C_DARK)

#  KPI-строка 
for val, lbl, xi in [('18 510', 'строк после очистки', 1.0),
                      ('38',     'дублей удалено',      4.5),
                      ('1',      'мусорный менеджер',   8.2),
                      ('27',     'уник. менеджеров',   11.1)]:
    add_text(slide, val, xi, 5.82, 2.0, 0.62,
             size=30, bold=True, color=C_TEAL, align=PP_ALIGN.CENTER)
    add_text(slide, lbl, xi, 6.44, 2.0, 0.38,
             size=11, color=C_GRAY, align=PP_ALIGN.CENTER)

print('Слайд 4 добавлен: Contacts')


Слайд 4 добавлен: Contacts


## Слайд 5 — Spend

In [58]:
slide = prs.slides.add_slide(BLANK)
bg_light(slide)
add_header_bar(slide, 'Очистка: Spend',
               'Рекламные расходы по дням — 20 779 строк исходно')

C_RED = RGBColor(0xC0, 0x39, 0x2B)
C_AMB = RGBColor(0xD3, 0x5A, 0x00)

BW, BH = 6.2, 1.92
CX1, CX2 = 0.35, 6.78
RY1, RY2 = 1.65, 3.73

#  Box 1: Что случилось 
add_rect(slide, CX1, RY1, BW, 0.36, C_DARK)
add_text(slide, 'Что случилось', CX1 + 0.1, RY1 + 0.03, BW - 0.15, 0.3,
         size=13, bold=True, color=C_WHITE)
add_bullet_box(slide, [
    '917 полных дублирующихся строк в выгрузке',
    'Поле date — тип object (строка), а не datetime',
    'spend / clicks — числа с запятой и символами (object)',
    '5 994 пропуска в campaign, 6 828 в adgroup и ad',
], CX1 + 0.12, RY1 + 0.42, BW - 0.2, BH - 0.5, size=12, color=C_DARK)

#  Box 2: Почему случилось 
add_rect(slide, CX2, RY1, BW, 0.36, C_AMB)
add_text(slide, 'Почему случилось', CX2 + 0.1, RY1 + 0.03, BW - 0.15, 0.3,
         size=13, bold=True, color=C_WHITE)
add_bullet_box(slide, [
    'Повторная выгрузка из рекл. кабинета дублирует записи',
    'Форматы даты и числа зависят от региональных настроек',
    'Кампании без структуры ad group → пропуски иерархии',
    'Органика и часть каналов не имеют расходов в системе',
], CX2 + 0.12, RY1 + 0.42, BW - 0.2, BH - 0.5, size=12, color=C_DARK)

#  Box 3: Если ничего не делать 
add_rect(slide, CX1, RY2, BW, 0.36, C_RED)
add_text(slide, 'Если ничего не делать', CX1 + 0.1, RY2 + 0.03, BW - 0.15, 0.3,
         size=13, bold=True, color=C_WHITE)
add_bullet_box(slide, [
    'Двойной учёт расходов → ROMI занижен',
    'Нельзя сделать groupby по дате (object != datetime)',
    'sum(spend) → ошибка: строку и число не сложить',
    'Пропуски в кампаниях → "Unknown" скрывает реальные данные',
], CX1 + 0.12, RY2 + 0.42, BW - 0.2, BH - 0.5, size=12, color=C_DARK)

#  Box 4: Что сделали 
add_rect(slide, CX2, RY2, BW, 0.36, C_TEAL)
add_text(slide, 'Что сделали', CX2 + 0.1, RY2 + 0.03, BW - 0.15, 0.3,
         size=13, bold=True, color=C_DARK)
add_bullet_box(slide, [
    'Удалено 917 дублей',
    'Даты преобразованы в datetime64[us]',
    'spend, clicks >> regex-очистка >> float64 / int64',
    'Пропуски campaign / adgroup / ad >> заполнены "Unknown"',
    'Результат: 19 862 строки, 14 источников трафика',
], CX2 + 0.12, RY2 + 0.42, BW - 0.2, BH - 0.5, size=12, color=C_DARK)

#  KPI-строка 
for val, lbl, xi in [('19 862', 'строк после очистки', 1.0),
                      ('917',    'дублей удалено',      4.5),
                      ('14',     'источников трафика',  8.2),
                      ('1 год',  'период данных',       11.1)]:
    add_text(slide, val, xi, 5.82, 2.0, 0.62,
             size=30, bold=True, color=C_TEAL, align=PP_ALIGN.CENTER)
    add_text(slide, lbl, xi, 6.44, 2.0, 0.38,
             size=11, color=C_GRAY, align=PP_ALIGN.CENTER)

print('Слайд 5 добавлен: Spend')


Слайд 5 добавлен: Spend


## Слайд 6 — Deals

In [59]:
slide = prs.slides.add_slide(BLANK)
bg_light(slide)
add_header_bar(slide, 'Очистка: Deals',
               'Основной датасет сделок — 21 595 строк исходно')

C_RED = RGBColor(0xC0, 0x39, 0x2B)
C_AMB = RGBColor(0xD3, 0x5A, 0x00)

BW, BH = 6.2, 1.92
CX1, CX2 = 0.35, 6.78
RY1, RY2 = 1.65, 3.73

#  Box 1: Что случилось 
add_rect(slide, CX1, RY1, BW, 0.36, C_DARK)
add_text(slide, 'Что случилось', CX1 + 0.1, RY1 + 0.03, BW - 0.15, 0.3,
         size=13, bold=True, color=C_WHITE)
add_bullet_box(slide, [
    '1 771 сделка с lost_reason = "дубликат" + 7 бизнес-дублей',
    'SLA хранился как datetime.time, а не число секунд',
    '215 уникальных значений уровня немецкого (рус.+лат. mix)',
    '13 стадий CRM — нет единой бизнес-воронки',
    'Суммы и города — строки с мусором и артефактами',
], CX1 + 0.12, RY1 + 0.42, BW - 0.2, BH - 0.5, size=11.5, color=C_DARK)

#  Box 2: Почему случилось 
add_rect(slide, CX2, RY1, BW, 0.36, C_AMB)
add_text(slide, 'Почему случилось', CX2 + 0.1, RY1 + 0.03, BW - 0.15, 0.3,
         size=13, bold=True, color=C_WHITE)
add_bullet_box(slide, [
    'Менеджеры вручную отмечали "дубликат" в CRM-поле',
    'CRM хранит SLA как время суток, а не длительность',
    'Отсутствие стандарта при вводе уровня языка',
    'Продажи эволюционировали → стадии переименовывались',
    'Ввод городов в свободном поле без валидации',
], CX2 + 0.12, RY1 + 0.42, BW - 0.2, BH - 0.5, size=11.5, color=C_DARK)

#  Box 3: Если ничего не делать 
add_rect(slide, CX1, RY2, BW, 0.36, C_RED)
add_text(slide, 'Если ничего не делать', CX1 + 0.1, RY2 + 0.03, BW - 0.15, 0.3,
         size=13, bold=True, color=C_WHITE)
add_bullet_box(slide, [
    'Дубли завышают количество сделок >> конверсия неверные',
    'SLA нечитаем >> нельзя считать среднее время обработки',
    '215 уровней владения языком>> анализ по языку невозможен',
    'Без воронки нет CR по стадиям и нельзя строить funnel',
    'JOIN с Contacts по мусорному источнику >> потеря данных',
], CX1 + 0.12, RY2 + 0.42, BW - 0.2, BH - 0.5, size=11.5, color=C_DARK)

#  Box 4: Что сделали 
add_rect(slide, CX2, RY2, BW, 0.36, C_TEAL)
add_text(slide, 'Что сделали', CX2 + 0.1, RY2 + 0.03, BW - 0.15, 0.3,
         size=13, bold=True, color=C_DARK)
add_bullet_box(slide, [
    'Удалено 1 778 дублей (1771 + 7 бизнес) >> 19 815 строк',
    'SLA >> int-секунды; sla_filled восстановлен (6 578 NaN)',
    '215 вариантов >> 7 стандартов (A1/A2/B1/B2/C1/C2/Unclear)',
    '13 стадий >> 4 группы: Marketing / Active Sales / Won / Lost',
    'Признаки обогащения: is_buyer, stage_group, deal_duration_days',
], CX2 + 0.12, RY2 + 0.42, BW - 0.2, BH - 0.5, size=11.5, color=C_DARK)

#  KPI-строка 
for val, lbl, xi in [('19 815', 'строк после очистки', 0.8),
                      ('1 778',  'дублей удалено',      4.1),
                      ('13 >> 4',   'стадий CRM',          7.2),
                      ('215 >> 7',  'уровней языка',       9.9),
                      ('4.2%',   'CR >> покупка',       12.3)]:
    add_text(slide, val, xi, 5.82, 1.7, 0.62,
             size=26, bold=True, color=C_TEAL, align=PP_ALIGN.CENTER)
    add_text(slide, lbl, xi, 6.44, 1.7, 0.38,
             size=10, color=C_GRAY, align=PP_ALIGN.CENTER)

print('Слайд 6 добавлен: Deals')


Слайд 6 добавлен: Deals


## Слайд 7 — Calls

In [60]:
slide = prs.slides.add_slide(BLANK)
bg_light(slide)
add_header_bar(slide, 'Очистка: Calls',
               'Данные о звонках менеджеров — 95 874 строки исходно')

C_RED = RGBColor(0xC0, 0x39, 0x2B)
C_AMB = RGBColor(0xD3, 0x5A, 0x00)

BW, BH = 6.2, 1.92
CX1, CX2 = 0.35, 6.78
RY1, RY2 = 1.65, 3.73

#  Box 1: Что случилось 
add_rect(slide, CX1, RY1, BW, 0.36, C_DARK)
add_text(slide, 'Что случилось', CX1 + 0.1, RY1 + 0.03, BW - 0.15, 0.3,
         size=13, bold=True, color=C_WHITE)
add_bullet_box(slide, [
    '3 275 содержательных дублей (95 874 → 92 599)',
    'Дата-поля хранились как object',
    'Колонка outgoing_call_status дублирует call_status',
    '7 913 записей с пересечением времени у менеджера',
    'NaN в contactid у 3 799 звонков (нет привязки)',
    'outgoing_call_status: Для Outbound звонков значения дублируют `call_status`',
    'scheduled_in_crm: Запланировано 134 звонка из 95874.',
    'tag: Колонка практически не заполнена.',
    'dialled_number: Колонка практически не заполнена.',
], CX1 + 0.12, RY1 + 0.42, BW - 0.2, BH - 0.5, size=12, color=C_DARK)

#  Box 2: Почему случилось 
add_rect(slide, CX2, RY1, BW, 0.36, C_AMB)
add_text(slide, 'Почему случилось', CX2 + 0.1, RY1 + 0.03, BW - 0.15, 0.3,
         size=13, bold=True, color=C_WHITE)
add_bullet_box(slide, [
    'CRM логирует несколько событий на одно TCP-соединение',
    'Тип данных дат при экспорте — строка (Zoho поведение)',
    'Дублирующая колонка — артефакт версионирования схемы',
    'Конференц-звонки / переводы → одновременные записи',
    'Холодные звонки не привязаны к контакту заранее',
], CX2 + 0.12, RY1 + 0.42, BW - 0.2, BH - 0.5, size=12, color=C_DARK)

#  Box 3: Если ничего не делать 
add_rect(slide, CX1, RY2, BW, 0.36, C_RED)
add_text(slide, 'Если ничего не делать', CX1 + 0.1, RY2 + 0.03, BW - 0.15, 0.3,
         size=13, bold=True, color=C_WHITE)
add_bullet_box(slide, [
    'COUNT звонков на 3.5% завышен — KPI активности неверен',
    'Datetime-операции (длительность, дельта) падают с ошибкой',
    'Дублирующая колонка даёт ложные сигналы при group by',
    'Пересечения маскируют реальную нагрузку менеджеров',
    'NaN в contactid → потеря части воронки при JOIN с Deals',
], CX1 + 0.12, RY2 + 0.42, BW - 0.2, BH - 0.5, size=12, color=C_DARK)

#  Box 4: Что сделали 
add_rect(slide, CX2, RY2, BW, 0.36, C_TEAL)
add_text(slide, 'Что сделали', CX2 + 0.1, RY2 + 0.03, BW - 0.15, 0.3,
         size=13, bold=True, color=C_DARK)
add_bullet_box(slide, [
    'drop_duplicates() → 3 275 дублей удалено → 92 599 строк',
    'call_start/end_time → datetime64[us]',
    'Удалена колонка outgoing_call_status (дубль)',
    '7 913 пересечений задокументированы, не удалялись',
    '110 звонков перепривязаны по mapping из Contacts',
    'is_successful: 72 590 / 92 599 (78.4%) успешных',
    'Удалены 4 мусорных столбца'
], CX2 + 0.12, RY2 + 0.42, BW - 0.2, BH - 0.5, size=11.5, color=C_DARK)

#  Предупреждение о пересечениях 
add_rect(slide, 0.35, 5.78, 12.63, 0.78, RGBColor(0xFF, 0xF3, 0xCD))
add_rect(slide, 0.35, 5.78, 0.06, 0.78, C_AMB)
add_text(slide,
         'Пересечения (7 913 строк): один менеджер = два одновременных звонка. '
         'Возможные причины: конференц-режим, переводы, баги CRM. '
         'Оставлены в данных — нужен отдельный анализ нагрузки.',
         0.55, 5.85, 12.2, 0.66, size=12, color=RGBColor(0x7B, 0x5B, 0x00))

#  KPI-строка 
for val, lbl, xi in [('92 599', 'строк после очистки', 1.0),
                      ('3 275',  'дублей удалено',      4.5),
                      ('7 913',  'пересечений (kept)',   8.2),
                      ('78.4%',  'звонков успешны',     11.1)]:
    add_text(slide, val, xi, 5.44, 2.0, 0.62,
             size=28, bold=True, color=C_TEAL, align=PP_ALIGN.CENTER)
    add_text(slide, lbl, xi, 6.06, 2.0, 0.38,
             size=11, color=C_GRAY, align=PP_ALIGN.CENTER)

print('Слайд 7 добавлен: Calls')


Слайд 7 добавлен: Calls


## Слайд 8 — Итоги очистки

In [61]:
slide = prs.slides.add_slide(BLANK)
bg_light(slide)
add_header_bar(slide, 'Итоги этапа очистки',
               'Данные верифицированы и готовы к анализу')

#  Сводная таблица (актуальные числа) 
headers = ['Датасет', 'До', 'Удалено', 'После', 'Ключевые преобразования']
rows_data = [
    ('Contacts', '18 548', '38',   '18 510', 'дубли, "False"-менеджер, datetime, Int64'),
    ('Spend',    '20 779', '917',  '19 862', 'дубли, datetime, числа, пропуски → Unknown'),
    ('Deals',    '21 595', '1 780','19 815', 'дубли, SLA→int, 215→7 яз., 13→4 стадий'),
    ('Calls',    '95 874', '3 275','92 599', 'дубли, datetime, пересечения задокументированы'),
]

col_x = [0.4, 2.85, 4.55, 6.1, 7.65]
col_w = [2.35, 1.6,  1.45, 1.45, 5.55]
ty = 1.68

for xi, wi, hdr in zip(col_x, col_w, headers):
    add_rect(slide, xi, ty, wi, 0.42, C_DARK)
    add_text(slide, hdr, xi + 0.05, ty + 0.05, wi - 0.1, 0.33,
             size=13, bold=True, color=C_WHITE)

for ri, row in enumerate(rows_data):
    row_y = ty + 0.42 + ri * 0.55
    bg = C_WHITE if ri % 2 == 0 else C_LIGHT
    add_rect(slide, 0.4, row_y, 12.88, 0.55, bg)
    for xi, wi, val in zip(col_x, col_w, row):
        add_text(slide, val, xi + 0.05, row_y + 0.08, wi - 0.1, 0.4,
                 size=12.5, color=C_DARK)

# Итого-строка
sum_y = ty + 0.42 + len(rows_data) * 0.55
add_rect(slide, 0.4, sum_y, 12.88, 0.42, C_TEAL)
add_text(slide, 'Итого', 0.45, sum_y + 0.06, 2.3, 0.32,
         size=13, bold=True, color=C_DARK)
add_text(slide, '156 796', 2.9, sum_y + 0.06, 1.55, 0.32,
         size=13, bold=True, color=C_DARK)
add_text(slide, '6 010', 4.6, sum_y + 0.06, 1.4, 0.32,
         size=13, bold=True, color=C_DARK)
add_text(slide, '150 786', 6.15, sum_y + 0.06, 1.4, 0.32,
         size=13, bold=True, color=C_DARK)
add_text(slide, f'−3.8% строк удалено  |  4 единых PKL-файла', 7.7, sum_y + 0.06, 5.4, 0.32,
         size=13, bold=True, color=C_DARK)

#  Выводы 
add_rect(slide, 0.4, 4.92, 12.88, 0.05, C_TEAL)
conclusions = [
    'Все 4 датасета связаны через единый ключ contact_id (Int64)',
    'Инженерные признаки: is_buyer, stage_group — основа воронки и юнит-экономики',
    'Данные верифицированы: типы, диапазоны, пропуски задокументированы',
    'Готовность к: EDA, продуктовой аналитике, стратегическим выводам',
]
add_bullet_box(slide, conclusions, 0.5, 5.1, 12.5, 2.1, size=13, color=C_DARK)

print('Слайд 8 добавлен: Итоги очистки')

Слайд 8 добавлен: Итоги очистки


## Слайд 9 — Ключевые бизнес-метрики

In [62]:
CLEANED_DIR = os.path.join('..', 'data', 'cleaned')
deals_s9    = pd.read_pickle(os.path.join(CLEANED_DIR, 'deals_clean.pkl'))
spend_s9    = pd.read_pickle(os.path.join(CLEANED_DIR, 'spend_clean.pkl'))
contacts_s9 = pd.read_pickle(os.path.join(CLEANED_DIR, 'contacts_clean.pkl'))

n_leads_s9       = len(deals_s9)
n_buyers_s9      = int(contacts_s9['is_buyer'].sum())
buyer_deals_s9   = deals_s9[deals_s9['is_buyer'] == 1]
n_transactions_s9 = len(buyer_deals_s9)                          # T — транзакции покупателей
revenue_s9       = buyer_deals_s9['initial_amount_paid'].sum()   # Rev — только подтверждённые покупатели
total_spend_s9   = spend_s9['spend'].sum()
cac_s9           = total_spend_s9 / n_buyers_s9
romi_s9          = (revenue_s9 - total_spend_s9) / total_spend_s9 * 100
cr_s9            = n_buyers_s9 / n_leads_s9 * 100
aov_s9           = revenue_s9 / n_buyers_s9                      # AOV = Rev / B (согласованно с 07)

slide = prs.slides.add_slide(BLANK)
bg_light(slide)
add_header_bar(slide, 'Ключевые бизнес-метрики',
               'Итоговые показатели за весь период анализа')

#  4 крупных KPI-карточки 
kpis = [
    (f'{romi_s9:,.0f}%',          'ROMI',    'Выручка / Бюджет × 100%',       C_GREEN),
    (f'{cr_s9:.1f}%',             'C1',      'Лид → Покупатель',              C_TEAL),
    (f'€ {cac_s9:,.0f}',          'CAC',     'Стоимость привлечения клиента', C_DARK),
    (f'€ {revenue_s9/1e6:.2f}M',  'Revenue', f'{n_buyers_s9:,} покупателей',  C_DARK),
]

kpi_w, kpi_h = 2.9, 2.5
for i, (val, label, sub, color) in enumerate(kpis):
    x = 0.35 + i * 3.24
    y = 1.65
    add_rect(slide, x, y, kpi_w, kpi_h, C_WHITE)
    add_rect(slide, x, y, kpi_w, 0.08, color)
    add_text(slide, val,   x + 0.15, y + 0.25, kpi_w - 0.3, 1.0,
             size=38, bold=True, color=color, align=PP_ALIGN.CENTER)
    add_text(slide, label, x + 0.15, y + 1.25, kpi_w - 0.3, 0.55,
             size=20, bold=True, color=C_DARK, align=PP_ALIGN.CENTER)
    add_text(slide, sub,   x + 0.15, y + 1.82, kpi_w - 0.3, 0.55,
             size=12, color=C_GRAY, align=PP_ALIGN.CENTER)

#  Воронка 
n_clicks_s9 = int(spend_s9['clicks'].sum())
conv_cl = n_leads_s9 / n_clicks_s9 * 100
conv_lb = n_buyers_s9 / n_leads_s9 * 100

stages_f = [
    (f'Клики\n{n_clicks_s9:,}',      4.8),
    (f'Лиды\n{n_leads_s9:,}',        3.4),
    (f'Покупатели\n{n_buyers_s9:,}', 2.0),
]
convs = [None, f'{conv_cl:.2f}%', f'{conv_lb:.1f}%']

fy = 4.42
for i, ((label, w), cv) in enumerate(zip(stages_f, convs)):
    fx = 0.5 + i * 4.2
    color_f = [C_DARK, RGBColor(0x10, 0x6E, 0xBF), C_GREEN][i]
    add_rect(slide, fx, fy, w, 0.85, color_f)
    add_text(slide, label, fx + 0.1, fy + 0.06, w - 0.2, 0.72,
             size=13, bold=True, color=C_WHITE, align=PP_ALIGN.CENTER)
    if cv:
        add_text(slide, f'→ {cv}', fx - 1.3, fy + 0.22, 1.2, 0.4,
                 size=14, bold=True, color=C_TEAL, align=PP_ALIGN.CENTER)

add_text(slide, 'Воронка продаж', 0.5, fy - 0.45, 13.0, 0.38,
         size=14, bold=True, color=C_DARK)
add_rect(slide, 0.5, fy - 0.08, 12.5, 0.04, C_TEAL)

#  Итоговая строка: Rev, T, AOV, CAC, Период 
add_rect(slide, 0.5, 5.65, 12.5, 0.05, C_TEAL)
add_text(slide,
         f'Бюджет: €{total_spend_s9:,.0f}   |   T: {n_transactions_s9:,} транзакций   |   '
         f'AOV: €{aov_s9:,.0f}   |   LTV/CAC: {aov_s9/cac_s9:.1f}   |   '
         f'Период: {deals_s9["created_time"].min().strftime("%b %Y")} – {deals_s9["created_time"].max().strftime("%b %Y")}',
         0.5, 5.78, 12.5, 0.45, size=13, color=C_GRAY, align=PP_ALIGN.CENTER)

print('Слайд 9 добавлен: Ключевые бизнес-метрики')


Слайд 9 добавлен: Ключевые бизнес-метрики


## Слайд 10 — Маркетинг: источники и ROMI

In [63]:
campaign_romi_s10 = pd.read_pickle(os.path.join(CLEANED_DIR, 'campaign_romi.pkl'))

source_s10 = (
    campaign_romi_s10.groupby('source', observed=True)
    .agg(leads=('deals_count', 'sum'), buyers=('buyers_count', 'sum'),
         revenue=('revenue', 'sum'), total_spend=('total_spend', 'sum'))
    .reset_index()
    .assign(
        romi=lambda x: ((x['revenue'] - x['total_spend']) / x['total_spend'].replace(0, np.nan) * 100).fillna(0).round(0),
        cac=lambda x: (x['total_spend'] / x['buyers'].replace(0, np.nan)).fillna(0).round(0),
        cr=lambda x: (x['buyers'] / x['leads'].replace(0, np.nan) * 100).fillna(0).round(1),
        aov=lambda x: (x['revenue'] / x['buyers'].replace(0, np.nan)).fillna(0).round(0),  # AOV = Rev / T
    )
    .sort_values('revenue', ascending=False)
    .head(8)
)

slide = prs.slides.add_slide(BLANK)
bg_light(slide)
add_header_bar(slide, 'Маркетинг: источники трафика',
               'ROMI, конверсия и CAC по каналам привлечения')

#  Заголовки таблицы — 9 колонок: Источник | Лиды | Покупат.    | CR% | Расходы | Rev | AOV | ROMI | CAC
hdrs_s10  = ['Источник', 'Лиды', 'Покупат.', 'CR%', 'Расходы €', 'Выручка €', 'AOV €', 'ROMI %', 'CAC €']
col_x_s10 = [0.35,  2.15,  3.25,    4.35,   5.25,       6.60,       8.05,    9.35,    10.90]
col_w_s10 = [1.75,  1.05,  1.05,    0.85,   1.30,       1.40,       1.25,    1.50,     1.60]

ty10 = 1.65
for xi, wi, hdr in zip(col_x_s10, col_w_s10, hdrs_s10):
    add_rect(slide, xi, ty10, wi, 0.38, C_DARK)
    add_text(slide, hdr, xi + 0.05, ty10 + 0.04, wi - 0.1, 0.3,
             size=12, bold=True, color=C_WHITE)

C_RED10 = RGBColor(0xF5, 0xD5, 0xD5)
for ri, row in source_s10.iterrows():
    ry = ty10 + 0.38 + list(source_s10.index).index(ri) * 0.52
    is_google = 'google' in str(row['source']).lower()
    bg = C_RED10 if is_google else (C_WHITE if list(source_s10.index).index(ri) % 2 == 0 else C_LIGHT)
    add_rect(slide, 0.35, ry, 12.88, 0.52, bg)
    romi_color = C_GREEN if row['romi'] > 500 else (C_DARK if row['romi'] > 0 else RGBColor(0xC0, 0x39, 0x2B))
    aov_val = f"€ {row['aov']:,.0f}" if row['aov'] > 0 else '—'
    vals = [str(row['source']), f"{row['leads']:,}", f"{row['buyers']:,}",
            f"{row['cr']:.1f}%", f"€ {row['total_spend']:,.0f}",
            f"€ {row['revenue']:,.0f}", aov_val,
            f"{row['romi']:,.0f}%", f"€ {row['cac']:,.0f}"]
    for xi, wi, val in zip(col_x_s10, col_w_s10, vals):
        c = romi_color if val == f"{row['romi']:,.0f}%" else C_DARK
        add_text(slide, val, xi + 0.05, ry + 0.08, wi - 0.1, 0.38, size=12, color=c)

#  Вывод-блок
n_rows = len(source_s10)
note_y = ty10 + 0.38 + n_rows * 0.52 + 0.15
add_rect(slide, 0.35, note_y, 12.88, 0.05, C_TEAL)
add_rect(slide, 0.35, note_y + 0.12, 12.88, 0.88, C_WHITE)
add_rect(slide, 0.35, note_y + 0.12, 0.06, 0.88, RGBColor(0xC0, 0x39, 0x2B))
add_text(slide,
         ' Google Ads — крупнейший источник лидов, но наименьший ROMI среди платных каналов: '
         'расходы почти равны выручке. Рекомендация: аудит ключевых слов, перенос бюджета '
         'в каналы с ROMI > 1000% (SMM, Webinar, Facebook).',
         0.55, note_y + 0.17, 12.5, 0.76, size=12, color=C_DARK)

print('Слайд 10 добавлен: Источники трафика и ROMI')


Слайд 10 добавлен: Источники трафика и ROMI


## Слайд 11 — Отдел продаж: воронка и менеджеры

In [64]:
deals_s11 = pd.read_pickle(os.path.join(CLEANED_DIR, 'deals_clean.pkl'))

mgr_s11 = (
    deals_s11[deals_s11['deal_owner_name'] != 'Unknown']
    .assign(buyer_revenue=lambda x: x['initial_amount_paid'] * x['is_buyer'])  # Rev = только покупатели
    .groupby('deal_owner_name', observed=True)
    .agg(total=('id', 'count'), buyers=('is_buyer', 'sum'),
         revenue=('buyer_revenue', 'sum'))
    .assign(cr=lambda x: (x['buyers'] / x['total'] * 100).round(1),
            aov=lambda x: (x['revenue'] / x['buyers'].replace(0, np.nan)).round(0))
    .sort_values('revenue', ascending=False)
    .head(8)
    .reset_index()
)

# Аномалии: Won/Paid с 0 оплатой
zero_mask_s11 = (deals_s11['stage_group'] == 'Won/Paid') & (deals_s11['initial_amount_paid'] == 0)
zero_by_mgr = deals_s11[zero_mask_s11].groupby('deal_owner_name', observed=True)['id'].count().sort_values(ascending=False).head(5)

# Длительность сделок Won vs Lost
won_med  = deals_s11.loc[deals_s11['stage_group'] == 'Won/Paid', 'deal_duration_days'].median()
lost_med = deals_s11.loc[deals_s11['stage_group'] == 'Lost', 'deal_duration_days'].median()

slide = prs.slides.add_slide(BLANK)
bg_light(slide)
add_header_bar(slide, 'Отдел продаж',
               'Рейтинг менеджеров и аномалии в данных')

#  Левая колонка: таблица менеджеров
add_rect(slide, 0.35, 1.62, 7.8, 0.05, C_TEAL)
add_text(slide, 'Топ-8 менеджеров по выручке', 0.35, 1.68, 7.8, 0.38,
         size=14, bold=True, color=C_DARK)

mgr_hdrs  = ['Менеджер', 'Сделок', 'Покупат. (T)', 'C1%', 'Выручка € (Rev)', 'AOV €']
mgr_cx    = [0.35, 2.85, 3.95, 5.10, 6.10, 7.35]
mgr_cw    = [2.40, 1.00, 1.05, 0.90, 1.15, 1.0]
mty = 2.12
for xi, wi, hdr in zip(mgr_cx, mgr_cw, mgr_hdrs):
    add_rect(slide, xi, mty, wi, 0.35, C_DARK)
    add_text(slide, hdr, xi + 0.04, mty + 0.04, wi - 0.08, 0.28,
             size=11, bold=True, color=C_WHITE)

for ri, row in mgr_s11.iterrows():
    ry = mty + 0.35 + ri * 0.46
    bg = C_WHITE if ri % 2 == 0 else C_LIGHT
    add_rect(slide, 0.35, ry, 8.0, 0.46, bg)
    vals = [str(row['deal_owner_name'])[:20], f"{row['total']:,}",
            f"{row['buyers']:,}", f"{row['cr']:.1f}%",
            f"{row['revenue']:,.0f}", f"{int(row['aov']) if not np.isnan(row['aov']) else '—'}"]
    for xi, wi, val in zip(mgr_cx, mgr_cw, vals):
        add_text(slide, val, xi + 0.04, ry + 0.07, wi - 0.08, 0.34, size=11, color=C_DARK)

#  Правая колонка: 3 инсайта
rx = 8.5
add_rect(slide, rx, 1.62, 4.6, 0.05, C_TEAL)

# Инсайт 1: сравнение длительностей
add_rect(slide, rx, 1.75, 4.6, 1.4, C_WHITE)
add_rect(slide, rx, 1.75, 0.06, 1.4, C_TEAL)
add_text(slide, 'Длительность сделок', rx + 0.15, 1.82, 4.3, 0.38,
         size=13, bold=True, color=C_DARK)
add_text(slide,
         f'Won/Paid: медиана {won_med:.0f} дн.\n'
         f'Lost: медиана {lost_med:.0f} дн.\n'
         f'Вывод: сделки > 14 дней — кандидаты на закрытие',
         rx + 0.15, 2.22, 4.3, 0.88, size=11, color=C_DARK)

# Инсайт 2: аномалии 0-оплата
C_WARN = RGBColor(0xFF, 0xF3, 0xCD)
add_rect(slide, rx, 3.25, 4.6, 1.5, C_WARN)
add_rect(slide, rx, 3.25, 0.06, 1.5, C_ORANGE)
top_zero = zero_by_mgr.index[0] if len(zero_by_mgr) else 'N/A'
top_zero_cnt = zero_by_mgr.iloc[0] if len(zero_by_mgr) else 0
add_text(slide, ' Won/Paid c нулевой оплатой', rx + 0.15, 3.32, 4.3, 0.38,
         size=13, bold=True, color=RGBColor(0x7B, 0x5B, 0x00))
add_text(slide,
         f'Всего: {zero_mask_s11.sum()} сделок\n'
         f'Лидер: {top_zero} — {top_zero_cnt} сделок\n'
         f'Требует аудита данных и CRM-валидации',
         rx + 0.15, 3.72, 4.3, 0.95, size=11, color=RGBColor(0x7B, 0x5B, 0x00))

# Инсайт 3: SLA
sla_med = deals_s11['sla_filled'].dropna().median()
add_rect(slide, rx, 4.85, 4.6, 1.35, C_WHITE)
add_rect(slide, rx, 4.85, 0.06, 1.35, C_GREEN)
add_text(slide, '⚡  SLA — скорость ответа', rx + 0.15, 4.92, 4.3, 0.38,
         size=13, bold=True, color=C_DARK)
add_text(slide,
         f'Медиана SLA: {sla_med:.0f} мин\n'
         f'Гипотеза H1: снижение SLA до Q1-уровня\n'
         f'→ конверсия C1 +10% (A/B тест)',
         rx + 0.15, 5.32, 4.3, 0.82, size=11, color=C_DARK)

print('Слайд 11 добавлен: Отдел продаж')


Слайд 11 добавлен: Отдел продаж


## Слайд 12 — Продукты и география

In [65]:
deals_s12 = pd.read_pickle(os.path.join(CLEANED_DIR, 'deals_clean.pkl'))

excl = ['Unknown', 'Find yourself in IT']
prod_s12 = (
    deals_s12[~deals_s12['product'].isin(excl)]
    .assign(buyer_revenue=lambda x: x['initial_amount_paid'] * x['is_buyer'])  # Rev = только покупатели
    .groupby('product', observed=True)
    .agg(count=('id', 'count'), buyers=('is_buyer', 'sum'),
         revenue=('buyer_revenue', 'sum'))
    .assign(cr=lambda x: (x['buyers'] / x['count'] * 100).round(1),
            aov=lambda x: (x['revenue'] / x['buyers'].replace(0, np.nan)).round(0))
    .sort_values('revenue', ascending=False)
    .head(7)
    .reset_index()
)

# Уровни языка
lang_s12 = (
    deals_s12[deals_s12['level_of_deutsch'] != 'Unknown']
    .groupby('level_of_deutsch', observed=True)
    .agg(count=('id', 'count'), buyers=('is_buyer', 'sum'))
    .assign(cr=lambda x: (x['buyers'] / x['count'] * 100).round(1))
    .reindex([c for c in ['A0','A1','A2','B1','B2','C1','C2']
              if c in deals_s12['level_of_deutsch'].unique()])
    .reset_index()
    .dropna(subset=['cr'])
)

# Топ-5 городов
city_s12 = (
    deals_s12[~deals_s12['city'].isin(['Unknown', '-', '', 'None'])]
    .groupby('city', observed=True)
    .agg(count=('id', 'count'), buyers=('is_buyer', 'sum'))
    .assign(cr=lambda x: (x['buyers'] / x['count'] * 100).round(1))
    .sort_values('count', ascending=False)
    .head(5)
    .reset_index()
)

slide = prs.slides.add_slide(BLANK)
bg_light(slide)
add_header_bar(slide, 'Продукты и география',
               'Популярность курсов, уровень языка и топ-5 городов')

#  Левая область: таблица продуктов
add_rect(slide, 0.35, 1.60, 8.2, 0.05, C_TEAL)
add_text(slide, 'Продукты по выручке (топ-7)', 0.35, 1.68, 8.2, 0.36,
         size=14, bold=True, color=C_DARK)

p_hdrs = ['Продукт', 'Сделок', 'Покупат. (T)', 'CR%', 'Выручка € (Rev)', 'AOV €']
p_cx   = [0.35, 3.40, 4.55, 5.75, 6.65, 7.95]
p_cw   = [2.95, 1.05, 1.10, 0.80, 1.20, 0.85]
pty = 2.10
for xi, wi, hdr in zip(p_cx, p_cw, p_hdrs):
    add_rect(slide, xi, pty, wi, 0.35, C_DARK)
    add_text(slide, hdr, xi + 0.04, pty + 0.04, wi - 0.08, 0.28,
             size=11, bold=True, color=C_WHITE)

for ri, row in prod_s12.iterrows():
    ry = pty + 0.35 + ri * 0.44
    bg = C_WHITE if ri % 2 == 0 else C_LIGHT
    add_rect(slide, 0.35, ry, 8.2, 0.44, bg)
    aov_val = f"{int(row['aov'])}" if not np.isnan(row['aov']) else '—'
    vals = [str(row['product'])[:30], f"{row['count']:,}",
            f"{row['buyers']:,}", f"{row['cr']:.1f}%",
            f"{row['revenue']:,.0f}", aov_val]
    for xi, wi, val in zip(p_cx, p_cw, vals):
        add_text(slide, val, xi + 0.04, ry + 0.07, wi - 0.08, 0.32, size=11, color=C_DARK)

#  Правая область: уровень языка + топ-городов
rx = 8.8

# Уровень языка → C1
add_rect(slide, rx, 1.60, 4.25, 0.05, C_TEAL)
add_text(slide, 'Конверсия по уровню немецкого', rx, 1.68, 4.25, 0.36,
         size=13, bold=True, color=C_DARK)

bar_max = lang_s12['cr'].max() if len(lang_s12) else 10
for i, row in lang_s12.iterrows():
    by = 2.12 + i * 0.44
    bw_full = 3.5
    bw = bw_full * row['cr'] / bar_max if bar_max else 0
    color_b = C_GREEN if row['cr'] >= 6 else (C_TEAL if row['cr'] >= 3 else RGBColor(0xC0, 0x39, 0x2B))
    add_rect(slide, rx, by, bw_full, 0.36, C_LIGHT)
    add_rect(slide, rx, by, max(bw, 0.05), 0.36, color_b)
    add_text(slide, row['level_of_deutsch'], rx + 0.05, by + 0.06, 0.45, 0.26,
             size=11, bold=True, color=C_WHITE)
    add_text(slide, f"{row['cr']:.1f}% ({row['count']:,} лидов)",
             rx + bw_full + 0.08, by + 0.06, 1.5, 0.26, size=11, color=C_DARK)

# Топ-5 городов
cy_start = 2.12 + len(lang_s12) * 0.44 + 0.25
add_rect(slide, rx, cy_start, 4.25, 0.05, C_TEAL)
add_text(slide, 'Топ-5 городов по числу лидов', rx, cy_start + 0.08, 4.25, 0.36,
         size=13, bold=True, color=C_DARK)
for i, row in city_s12.iterrows():
    cy = cy_start + 0.5 + i * 0.44
    bw_full = 3.5
    bw = bw_full * row['count'] / city_s12['count'].max()
    cr_color = C_GREEN if row['cr'] >= 6 else (C_ORANGE if row['cr'] >= 3 else RGBColor(0xC0, 0x39, 0x2B))
    add_rect(slide, rx, cy, bw_full, 0.36, C_LIGHT)
    add_rect(slide, rx, cy, max(bw, 0.05), 0.36, C_DARK)
    add_text(slide, str(row['city'])[:14], rx + 0.05, cy + 0.06, 1.7, 0.26,
             size=11, bold=True, color=C_WHITE)
    add_text(slide, f"{row['count']:,} | CR {row['cr']:.1f}%",
             rx + bw_full + 0.08, cy + 0.06, 1.5, 0.26, size=11, color=cr_color)

print('Слайд 12 добавлен: Продукты и география')


Слайд 12 добавлен: Продукты и география


## Слайд 13 — Карта продаж по Европе

In [66]:
MAP_IMG = os.path.join('..', 'reports', 'map_europe.png')

if not os.path.exists(MAP_IMG):
    print(f' Файл не найден: {MAP_IMG}')
else:
    slide_map = prs.slides.add_slide(BLANK)
    bg_light(slide_map)
    add_header_bar(slide_map, 'Распределение контактов по городам Европы', '')

    slide_map.shapes.add_picture(
        MAP_IMG,
        PptxInches(0.20), PptxInches(1.15),
        width=PptxInches(12.90)
    )
    print('✓ Слайд 13 (Карта) добавлен')

✓ Слайд 13 (Карта) добавлен


## Слайд 14 — Продуктовая аналитика: Юнит-экономика

In [67]:
# Загрузка данных из product_analytics
report_data = pd.read_pickle(os.path.join(CLEANED_DIR, 'report_data.pkl'))
gkpi        = report_data['global_kpi']
sens_df     = report_data['sens_df']
hypotheses  = report_data.get('recommended_hypotheses', [])

slide = prs.slides.add_slide(BLANK)
bg_light(slide)
add_header_bar(slide, "Рычаги роста и A/B Гипотезы", "Анализ чувствительности воронки")

# Левая часть - Таблица чувствительности (Top-3)
add_rect(slide, 0.5, 1.8, 6.0, 4.5, C_WHITE)
add_text(slide, "Топ рычагов (Impact на Прибыль):", 0.7, 2.0, 5.6, 0.4, size=18, bold=True)

# Отрисовка мини-таблицы (3 строки)
y_off = 2.6
headers = ["Параметр", "Эффект +1%", "Приоритет"]
add_rect(slide, 0.7, y_off, 5.6, 0.4, C_DARK)
for i, txt in enumerate(headers):
    add_text(slide, txt, 0.7 + (i*1.8), y_off, 1.8, 0.4, size=12, color=C_WHITE, bold=True)

# Имена колонок в sens_df
param_col = 'Параметр' if 'Параметр' in sens_df.columns else sens_df.columns[0]
val_col   = 'Прирост Прибыли, €' if 'Прирост Прибыли, €' in sens_df.columns else sens_df.columns[1]

for idx, row in sens_df.head(3).iterrows():
    y_off += 0.5
    add_text(slide, f"{row[param_col]}", 0.7, y_off, 2.0, 0.4, size=14)
    # Пытаемся привести к числу перед форматированием
    try:
        val_numeric = float(str(row[val_col]).replace('€','').replace(',','').strip())
        val_str = f"+{val_numeric:,.0f} €"
    except:
        val_str = f"{row[val_col]}"
    
    add_text(slide, val_str, 2.6, y_off, 1.8, 0.4, size=14, color=C_GREEN, bold=True)
    add_text(slide, "Высокий" if idx==0 else "Средний", 4.4, y_off, 1.8, 0.4, size=14)

# Правая часть - Гипотезы
add_rect(slide, 7.0, 1.8, 5.8, 4.5, C_WHITE)
add_text(slide, "Прокси-гипотезы (Test speed < 14 дней):", 7.2, 2.0, 5.4, 0.4, size=18, bold=True)

bullet_points = []
for h in hypotheses[:4]:
    p_name = h.get('Продукт', 'Общая')
    m_name = h.get('Прокси-метрика', 'CR')
    days_v = h.get('Дней на тест', 0)
    bullet_points.append(f"{p_name}: {m_name} ({days_v:.0f} дн.)")

if not bullet_points:
    bullet_points = [
        "H5 (Лид-магнит): Тест CR в скачивание (7 дн.)",
        "H4 (Авто-касания): Тест Reply Rate (10 дн.)"
    ]

add_bullet_box(slide, bullet_points, 7.2, 2.6, 5.4, 3.5, size=15)

print("Слайд 14 добавлен: Рычаги и Гипотезы")

Слайд 14 добавлен: Рычаги и Гипотезы


## Слайд 15 — Анализ чувствительности

In [68]:
# Слайд 15 - Расширенный анализ чувствительности (UA, C1, AOV, APC, CPA)
slide = prs.slides.add_slide(BLANK)
bg_light(slide)
add_header_bar(slide, 'Точки роста: анализ чувствительности по ТОП-продуктам (UA, C1, AOV, APC, CPA)',
               'Как изменение конкретных метрик влияет на маржинальную прибыль (CM)')

# Заголовки таблицы 
s_hdrs = ['Сценарий', 'UA', 'C1, %', 'APC', 'AOV, €', 'LTV', 'CPA, €', 'CM, €', 'ΔCM, €']
s_cx   = [0.35, 2.70, 3.80, 4.80, 5.75, 7.15, 8.35, 9.65, 11.20]
s_cw   = [2.25, 1.00, 0.90, 0.90, 1.30, 1.15, 1.20, 1.45, 1.55]

ty_s = 1.62
for xi, wi, hdr in zip(s_cx, s_cw, s_hdrs):
    add_rect(slide, xi, ty_s, wi, 0.35, C_DARK)
    add_text(slide, hdr, xi + 0.04, ty_s + 0.04, wi - 0.08, 0.28,
             size=9.5, bold=True, color=C_WHITE)

# Используем sens_df из report_data (только один топовый продукт для детальности, либо первые 10 строк)
s_df_filtered = sens_df.head(12).copy() # База + 5 сценариев для топ-2 продуктов

for ri in range(len(s_df_filtered)):
    row = s_df_filtered.iloc[ri]
    ry = ty_s + 0.35 + ri * 0.38
    
    is_base = row['Сценарий'] == 'Факт (база)'
    bg = C_LIGHT if not is_base else C_WHITE
    add_rect(slide, 0.35, ry, 12.88, 0.38, bg)
    
    # Визуальный разделитель продуктов
    if is_base and ri > 0:
        add_rect(slide, 0.35, ry, 12.88, 0.03, C_TEAL)
        # Добавим название продукта поверх если это база
        add_text(slide, f"Продукт: {row['Продукт']}", 0.35, ry - 0.25, 4.0, 0.25, size=9, bold=True, color=C_TEAL)

    cm_v  = row.get('CM, €', 0)
    dcm_v = row.get('ΔCM, €', float('nan'))

    def fmt_nan(v, fmt):
        return '—' if v != v else fmt.format(v)

    vals = [
        str(row['Сценарий']),
        f"{row.get('UA', 0):,.0f}",
        f"{row.get('C1', 0):.2%}",
        f"{row.get('APC', 0):.2f}",
        f"{row.get('AOV', 0):,.0f}",
        f"{row.get('LTV', 0):.2f}",
        f"{row.get('CPA (LTC)', 0):.2f}",
        f"{cm_v:,.0f}",
        fmt_nan(dcm_v, '{:+,.0f}')
    ]

    for ci, (xi, wi, val) in enumerate(zip(s_cx, s_cw, vals)):
        color = C_DARK
        if ci == 0 and is_base: color = C_TEAL
        if ci == 7 and cm_v < 0: color = RGBColor(0xC0, 0x39, 0x2B)
        if ci == 8 and dcm_v > 0: color = C_GREEN
        
        add_text(slide, val, xi + 0.04, ry + 0.04, wi - 0.08, 0.30, 
                 size=9.5, bold=(is_base and ci == 0), color=color)

# Выводы
add_rect(slide, 0.35, 6.6, 12.88, 0.05, C_TEAL)
add_bullet_box(slide, [
    'APC (T/B) — частота покупок. Рост APC на 10% влияет на маржу так же сильно, как рост AOV.',
    'CPA (Затраты на лид) — снижение на 10% дает экономию бюджета без снижения Revenue.',
    'LTV = C1 * APC * AOV — совокупный показатель качества продукта и маркетинга.',
], 0.55, 6.8, 12.5, 0.8, size=12, color=C_DARK)

print('Слайд 15 (Расширенная чувствительность) добавлен')


Слайд 15 (Расширенная чувствительность) добавлен


## Слайд 16 — A/B Гипотезы

In [69]:
# --- Слайд 16: Обоснование стратегии тестирования (Proxy vs Money) ---
prod_hyp = report_data.get('product_hypotheses_names', [])
rec_hyp  = report_data.get('recommended_hypotheses', [])

slide_16 = prs.slides.add_slide(prs.slide_layouts[6])
bg_light(slide_16)
add_header_bar(slide_16, "A/B Тестирование: Скорость и Прокси-метрики", "Почему классические тесты на Продажи здесь не работают")

# Левая колонка - Проблема
add_rect(slide_16, 0.5, 1.8, 5.8, 5.0, C_WHITE)
add_text(slide_16, "Проблема: Окно принятия решения", 0.7, 2.0, 5.4, 0.4, size=20, bold=True, color=C_RED)

problem_list = [
    "Цикл сделки составляет 3-6 месяцев (от лида до оплаты)",
    "Для теста C1 (Конверсия в оплату) нужно ждать завершения цикла",
    "При текущем трафике (4.4% CR) статистическая значимость\nпо деньгам достижима через 1.5 - 2 года",
    "Тестировать 'гипотезы роста' по финальной выручке — путь к застою"
]
add_bullet_box(slide_16, problem_list, 0.7, 2.6, 5.4, 4.0, size=16)

# Правая колонка - Решение (H4, H5)
add_rect(slide_16, 6.8, 1.8, 6.0, 5.0, C_WHITE)
add_text(slide_16, "Решение: Прокси-метрики (H4, H5)", 7.0, 2.0, 5.6, 0.4, size=20, bold=True, color=C_GREEN)

solution_list = [
    "H5 (Лид-магнит): Тестируем CR в 'скачивание пользы'",
    "Результат виден за 7-10 дней (быстрый поток данных)",
    "H4 (Авто-касания): Тестируем Open Rate и Reply Rate",
    "Валидация интереса аудитории без ожидания оплат",
    "Корреляция: Высокий интерес к магниту = выше CR в продажу"
]
add_bullet_box(slide_16, solution_list, 7.0, 2.6, 5.6, 4.0, size=16)

# Футер-вывод
add_rect(slide_16, 0.5, 6.9, 12.3, 0.5, C_DARK)
add_text(slide_16, "СТРАТЕГИЯ: Фокус на 'верх воронки' (Proxy) позволяет проводить 2 теста в месяц вместо 1 в год.", 
         0.6, 6.95, 12.1, 0.4, size=14, bold=True, color=C_WHITE, align=PP_ALIGN.CENTER)

## Слайд 13 — Выводы и рекомендации

In [70]:
slide = prs.slides.add_slide(BLANK)
bg_light(slide)
add_header_bar(slide, 'Выводы и рекомендации', 'По результатам анализа данных за весь период')

#  Левая колонка: ключевые выводы 
findings = [
    (' Google Ads',
     'Максимальный объём лидов, но наименьший ROMI среди платных каналов. '
     'Требует аудита ключевых слов и перераспределения бюджета.'),
    (' Воронка продаж',
     'Два узких места: Клики→Лиды (релевантность объявлений) и Лиды→Покупатели '
     '(скорость обработки и скрипты продаж).'),
    (' AOV / CAC ≈ 1.06',
     'Бизнес-модель "одной продажи" (T/B ≈ 1.01). '
     'Фактически нулевой запас прочности — оптимизация конверсии обязательна.'),
    (' Аномалия CRM',
     '14+ сделок Won/Paid с нулевой оплатой — требуется проверка '
     'CRM-процесса и валидации оплат.'),
    (' Целевая аудитория',
     'Уровни B2–C1 показывают наилучшую конверсию. '
     'A0–A1 — кандидаты для отдельной воронки или исключения из таргетинга.'),
]

col_l = 0.4
for i, (title, body) in enumerate(findings):
    fy = 1.55 + i * 1.02
    add_rect(slide, col_l, fy, 6.2, 0.95, C_WHITE)
    add_rect(slide, col_l, fy, 0.06, 0.95, C_TEAL)
    add_text(slide, title, col_l + 0.15, fy + 0.05, 5.95, 0.30,
             size=12, bold=True, color=C_DARK)
    add_text(slide, body, col_l + 0.15, fy + 0.35, 5.95, 0.55,
             size=11, color=C_GRAY)

#  Правая колонка: приоритетные действия 
rx = 7.0
add_rect(slide, rx, 1.50, 5.95, 0.40, C_DARK)
add_text(slide, 'Приоритетные действия', rx + 0.15, 1.55, 5.65, 0.30,
         size=14, bold=True, color=C_WHITE)

actions = [
    ('1', 'Перераспределить бюджет',
     'Снизить долю Google Ads, увеличить бюджет на каналы с ROMI > 3 000.',
     C_ORANGE),
    ('2', 'H4: WhatsApp-бот (7.6 дн.)',
     'Автоматический первый контакт ≤ 2 мин после регистрации лида.\n'
     'Прокси-метрика: "Успешный контакт" 50% → 70%.',
     C_TEAL),
    ('3', 'H5: Lead-Magnet (10.5 дн.)',
     'Бесплатный интенсив вместо продажи "в лоб".\n'
     'Прокси: "Запись на интенсив" 30% → 45%. Снимает барьер 1-й оплаты.',
     C_GREEN),
    ('4', 'Таргетировать B2–C1',
     'Создать отдельные рекламные кампании для носителей B2/C1.\n'
     'A0–A1 тестировать в отдельной воронке с базовым курсом.',
     RGBColor(0x8E, 0x44, 0xAD)),
]

for i, (num, title, body, accent) in enumerate(actions):
    ay = 2.02 + i * 1.22
    add_rect(slide, rx, ay, 5.95, 1.15, C_WHITE)
    add_rect(slide, rx, ay, 0.45, 1.15, accent)
    add_text(slide, num, rx + 0.09, ay + 0.30, 0.28, 0.50,
             size=20, bold=True, color=C_WHITE)
    add_text(slide, title, rx + 0.60, ay + 0.08, 5.20, 0.32,
             size=13, bold=True, color=C_DARK)
    add_text(slide, body, rx + 0.60, ay + 0.42, 5.20, 0.65,
             size=11, color=C_GRAY)

print('Слайд 17 добавлен: Выводы и рекомендации')


Слайд 17 добавлен: Выводы и рекомендации


## Сохранение PPTX

In [71]:
OUT_DIR = os.path.join('..', 'reports')
os.makedirs(OUT_DIR, exist_ok=True)

out_path = os.path.join(OUT_DIR, 'crm_analytics_presentation.pptx')
prs.save(out_path)

size_kb = os.path.getsize(out_path) / 1024
print(f' Презентация сохранена: {out_path}')
print(f'   Слайдов: {len(prs.slides)}   |   Размер: {size_kb:.1f} КБ')

 Презентация сохранена: ../reports/crm_analytics_presentation.pptx
   Слайдов: 17   |   Размер: 528.3 КБ
